# Enhanced Business Analytics: Advanced Vendor Performance Insights

## Business Context
This notebook provides advanced analytics for vendor performance optimization, pricing strategies, and inventory management in the retail/wholesale industry.

### Key Business Objectives:
1. **Vendor Selection for Profitability** - Identify high-performing vendors
2. **Product Pricing Optimization** - Analyze pricing strategies and margins
3. **Inventory Management** - Optimize stock turnover and reduce holding costs
4. **Supply Chain Risk Mitigation** - Assess vendor dependency and concentration risks
5. **Operational Efficiency** - Improve freight costs and bulk purchasing strategies

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from scipy.stats import ttest_ind, pearsonr
import scipy.stats as stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

In [ ]:
# Connect to database and load data
conn = sqlite3.connect('inventory.db')

# Load vendor summary data
df = pd.read_sql_query("""
    SELECT * FROM vendor_sales_summary 
    WHERE TotalSalesDollars > 0 
    AND TotalPurchaseDollars > 0
    AND ProfitMargin BETWEEN -100 AND 100
""", conn)

# Load raw data for additional analysis
sales_raw = pd.read_sql_query("SELECT * FROM sales LIMIT 100000", conn)
purchases_raw = pd.read_sql_query("SELECT * FROM purchases LIMIT 100000", conn)
vendor_invoice_raw = pd.read_sql_query("SELECT * FROM vendor_invoice", conn)

print(f"Vendor Summary: {df.shape}")
print(f"Sales Sample: {sales_raw.shape}")
print(f"Purchases Sample: {purchases_raw.shape}")
print(f"Vendor Invoice: {vendor_invoice_raw.shape}")

In [ ]:
# Advanced Feature Engineering
def create_advanced_features(df):
    """Create advanced business metrics"""
    
    # Revenue-based metrics
    df['RevenuePerUnit'] = df['TotalSalesDollars'] / df['TotalSalesQuantity']
    df['CostPerUnit'] = df['TotalPurchaseDollars'] / df['TotalPurchaseQuantity']
    df['ProfitPerUnit'] = df['RevenuePerUnit'] - df['CostPerUnit']
    
    # Efficiency metrics
    df['InventoryEfficiency'] = df['TotalSalesQuantity'] / (df['TotalPurchaseQuantity'] + 1)
    df['FreightEfficiency'] = df['FreightCost'] / df['TotalPurchaseDollars']
    df['PricingEfficiency'] = df['TotalSalesDollars'] / (df['TotalPurchaseDollars'] + df['FreightCost'])
    
    # Risk metrics
    df['VendorDependency'] = df.groupby('VendorName')['TotalSalesDollars'].transform('sum') / df['TotalSalesDollars'].sum()
    df['BrandConcentration'] = df.groupby('Brand')['TotalSalesDollars'].transform('sum') / df['TotalSalesDollars'].sum()
    
    # Performance categories
    df['PerformanceCategory'] = pd.cut(df['TotalSalesDollars'], 
                                       bins=[0, df['TotalSalesDollars'].quantile(0.33), 
                                             df['TotalSalesDollars'].quantile(0.66), float('inf')],
                                       labels=['Low', 'Medium', 'High'])
    
    # Handle infinite values
    df.replace([np.inf, -np.inf], 0, inplace=True)
    df.fillna(0, inplace=True)
    
    return df

df_enhanced = create_advanced_features(df.copy())
print(f"Enhanced dataset created with {df_enhanced.shape[1]} features")
df_enhanced.head()

## 1. Advanced Vendor Segmentation Analysis

In [ ]:
# Vendor Clustering for Strategic Segmentation
def vendor_segmentation(df):
    """Perform vendor clustering for strategic segmentation"""
    
    # Aggregate vendor-level metrics
    vendor_metrics = df.groupby('VendorName').agg({
        'TotalSalesDollars': 'sum',
        'TotalPurchaseDollars': 'sum',
        'GrossProfit': 'sum',
        'ProfitMargin': 'mean',
        'StockTurnover': 'mean',
        'FreightCost': 'sum',
        'TotalPurchaseQuantity': 'sum',
        'TotalSalesQuantity': 'sum'
    }).reset_index()
    
    # Create features for clustering
    vendor_metrics['SalesToPurchaseRatio'] = vendor_metrics['TotalSalesDollars'] / vendor_metrics['TotalPurchaseDollars']
    vendor_metrics['FreightRatio'] = vendor_metrics['FreightCost'] / vendor_metrics['TotalPurchaseDollars']
    vendor_metrics['Profitability'] = vendor_metrics['GrossProfit'] / vendor_metrics['TotalSalesDollars']
    
    # Select features for clustering
    features = ['TotalSalesDollars', 'ProfitMargin', 'StockTurnover', 'SalesToPurchaseRatio', 'FreightRatio']
    X = vendor_metrics[features].fillna(0)
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Perform K-means clustering
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    vendor_metrics['Cluster'] = kmeans.fit_predict(X_scaled)
    
    # Analyze clusters
    cluster_analysis = vendor_metrics.groupby('Cluster').agg({
        'VendorName': 'count',
        'TotalSalesDollars': ['mean', 'sum'],
        'ProfitMargin': 'mean',
        'StockTurnover': 'mean',
        'SalesToPurchaseRatio': 'mean',
        'FreightRatio': 'mean'
    }).round(2)
    
    return vendor_metrics, cluster_analysis, kmeans, scaler

vendor_segments, cluster_analysis, kmeans_model, scaler_model = vendor_segmentation(df_enhanced)

print("=== VENDOR SEGMENTATION RESULTS ===")
print(cluster_analysis)

# Define cluster characteristics
cluster_names = {
    0: "High Performers",
    1: "Efficient Operators", 
    2: "Low Performers",
    3: "Premium Specialists"
}

vendor_segments['Segment'] = vendor_segments['Cluster'].map(cluster_names)
print(f"\nVendor segments created: {vendor_segments['Segment'].value_counts().to_dict()}")

In [ ]:
# Visualize Vendor Segments
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Vendor Segmentation Analysis', fontsize=16, fontweight='bold')

# Scatter plot: Sales vs Profit Margin
colors = ['red', 'blue', 'green', 'orange']
for i, segment in enumerate(vendor_segments['Segment'].unique()):
    segment_data = vendor_segments[vendor_segments['Segment'] == segment]
    axes[0,0].scatter(segment_data['TotalSalesDollars'], segment_data['ProfitMargin'], 
                      c=colors[i], label=segment, alpha=0.7, s=50)
    
axes[0,0].set_xlabel('Total Sales Dollars ($)')
axes[0,0].set_ylabel('Profit Margin (%)')
axes[0,0].set_title('Sales vs Profit Margin by Segment')
axes[0,0].legend()
axes[0,0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
axes[0,0].grid(True, alpha=0.3)

# Box plot: Profit Margin by Segment
segment_profit = [vendor_segments[vendor_segments['Segment'] == seg]['ProfitMargin'] 
                  for seg in vendor_segments['Segment'].unique()]
axes[0,1].boxplot(segment_profit, labels=vendor_segments['Segment'].unique())
axes[0,1].set_title('Profit Margin Distribution by Segment')
axes[0,1].set_ylabel('Profit Margin (%)')
axes[0,1].grid(True, alpha=0.3)

# Bar plot: Vendor count by segment
vendor_counts = vendor_segments['Segment'].value_counts()
axes[1,0].bar(vendor_counts.index, vendor_counts.values, color=['red', 'blue', 'green', 'orange'])
axes[1,0].set_title('Number of Vendors by Segment')
axes[1,0].set_ylabel('Number of Vendors')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].grid(True, alpha=0.3)

# Radar chart for segment characteristics
segment_means = vendor_segments.groupby('Segment')[['ProfitMargin', 'StockTurnover', 
                                                    'SalesToPurchaseRatio', 'FreightRatio']].mean()
    
# Normalize for radar chart
normalized_data = (segment_means - segment_means.min()) / (segment_means.max() - segment_means.min())
    
angles = np.linspace(0, 2 * np.pi, len(normalized_data.columns), endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

ax_radar = plt.subplot(2, 2, 4, projection='polar')
for i, segment in enumerate(normalized_data.index):
    values = normalized_data.loc[segment].tolist()
    values += values[:1]
    ax_radar.plot(angles, values, 'o-', linewidth=2, label=segment)
    ax_radar.fill(angles, values, alpha=0.25)
    
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(normalized_data.columns)
ax_radar.set_title('Segment Characteristics (Normalized)')
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.show()

## 2. Time Series and Trend Analysis

In [ ]:
# Time Series Analysis from Raw Data
def time_series_analysis(sales_data, purchase_data):
    """Analyze temporal trends in sales and purchases"""
    
    # Convert dates
    sales_data['SalesDate'] = pd.to_datetime(sales_data['SalesDate'])
    purchase_data['PODate'] = pd.to_datetime(purchase_data['PODate'])
    
    # Monthly aggregation
    sales_monthly = sales_data.groupby(sales_data['SalesDate'].dt.to_period('M')).agg({
        'SalesDollars': 'sum',
        'SalesQuantity': 'sum',
        'VendorNo': 'nunique'
    }).reset_index()
    
    purchase_monthly = purchase_data.groupby(purchase_data['PODate'].dt.to_period('M')).agg({
        'Dollars': 'sum',
        'Quantity': 'sum',
        'VendorNumber': 'nunique'
    }).reset_index()
    
    # Calculate trends
    sales_monthly['SalesGrowth'] = sales_monthly['SalesDollars'].pct_change() * 100
    purchase_monthly['PurchaseGrowth'] = purchase_monthly['Dollars'].pct_change() * 100
    
    return sales_monthly, purchase_monthly

sales_monthly, purchase_monthly = time_series_analysis(sales_raw, purchases_raw)

print("=== TIME SERIES ANALYSIS ===")
print(f"Sales data period: {sales_monthly['SalesDate'].min()} to {sales_monthly['SalesDate'].max()}")
print(f"Purchase data period: {purchase_monthly['PODate'].min()} to {purchase_monthly['PODate'].max()}")
print(f"Average monthly sales growth: {sales_monthly['SalesGrowth'].mean():.2f}%")
print(f"Average monthly purchase growth: {purchase_monthly['PurchaseGrowth'].mean():.2f}%")

In [ ]:
# Visualize Time Series Trends
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Time Series Analysis: Sales and Purchase Trends', fontsize=16, fontweight='bold')

# Sales trend
axes[0,0].plot(sales_monthly['SalesDate'].astype(str), sales_monthly['SalesDollars'], 
              marker='o', linewidth=2, markersize=6)
axes[0,0].set_title('Monthly Sales Trend')
axes[0,0].set_ylabel('Sales Dollars ($)')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].grid(True, alpha=0.3)
axes[0,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Purchase trend
axes[0,1].plot(purchase_monthly['PODate'].astype(str), purchase_monthly['Dollars'], 
              marker='s', linewidth=2, markersize=6, color='orange')
axes[0,1].set_title('Monthly Purchase Trend')
axes[0,1].set_ylabel('Purchase Dollars ($)')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].grid(True, alpha=0.3)
axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Growth rates
axes[1,0].plot(sales_monthly['SalesDate'].astype(str), sales_monthly['SalesGrowth'], 
              marker='o', linewidth=2, markersize=6, label='Sales Growth')
axes[1,0].plot(purchase_monthly['PODate'].astype(str), purchase_monthly['PurchaseGrowth'], 
              marker='s', linewidth=2, markersize=6, label='Purchase Growth')
axes[1,0].set_title('Monthly Growth Rates')
axes[1,0].set_ylabel('Growth Rate (%)')
axes[1,0].tick_params(axis='x', rotation=45)
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)
axes[1,0].axhline(y=0, color='black', linestyle='--', alpha=0.5)

# Vendor activity
axes[1,1].plot(sales_monthly['SalesDate'].astype(str), sales_monthly['VendorNo'], 
              marker='o', linewidth=2, markersize=6, label='Active Vendors (Sales)')
axes[1,1].plot(purchase_monthly['PODate'].astype(str), purchase_monthly['VendorNumber'], 
              marker='s', linewidth=2, markersize=6, label='Active Vendors (Purchases)')
axes[1,1].set_title('Monthly Vendor Activity')
axes[1,1].set_ylabel('Number of Active Vendors')
axes[1,1].tick_params(axis='x', rotation=45)
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Advanced Risk Analysis

In [ ]:
# Supply Chain Risk Assessment
def supply_chain_risk_analysis(df):
    """Assess supply chain risks and vendor dependency"""
    
    # Calculate vendor concentration metrics
    vendor_sales = df.groupby('VendorName')['TotalSalesDollars'].sum().sort_values(ascending=False)
    total_sales = vendor_sales.sum()
    
    # Concentration ratios
    cr4 = (vendor_sales.head(4).sum() / total_sales) * 100  # Top 4 vendors
    cr10 = (vendor_sales.head(10).sum() / total_sales) * 100  # Top 10 vendors
    hhi = (vendor_sales / total_sales * 100).pow(2).sum()  # Herfindahl-Hirschman Index
    
    # Risk categories
    if hhi < 1000:
        market_concentration = "Low"
    elif hhi < 1800:
        market_concentration = "Medium"
    else:
        market_concentration = "High"
    
    # Vendor risk scoring
    vendor_risk = df.groupby('VendorName').agg({
        'TotalSalesDollars': 'sum',
        'ProfitMargin': 'mean',
        'StockTurnover': 'mean',
        'FreightCost': 'sum'
    }).reset_index()
    
    # Calculate risk scores (0-100, higher = more risky)
    vendor_risk['ConcentrationRisk'] = (vendor_risk['TotalSalesDollars'] / total_sales) * 100
    vendor_risk['MarginRisk'] = np.where(vendor_risk['ProfitMargin'] < 10, 50, 0)  # Low margin risk
    vendor_risk['TurnoverRisk'] = np.where(vendor_risk['StockTurnover'] < 0.5, 30, 0)  # Low turnover risk
    vendor_risk['FreightRisk'] = (vendor_risk['FreightCost'] / vendor_risk['TotalSalesDollars']) * 100
    
    vendor_risk['TotalRiskScore'] = vendor_risk[['ConcentrationRisk', 'MarginRisk', 
                                                'TurnoverRisk', 'FreightRisk']].sum(axis=1)
    
    vendor_risk['RiskCategory'] = pd.cut(vendor_risk['TotalRiskScore'], 
                                        bins=[0, 20, 40, 60, 100], 
                                        labels=['Low', 'Medium', 'High', 'Critical'])
    
    return {
        'concentration_metrics': {'CR4': cr4, 'CR10': cr10, 'HHI': hhi, 'Level': market_concentration},
        'vendor_risk_scores': vendor_risk
    }

risk_analysis = supply_chain_risk_analysis(df_enhanced)

print("=== SUPPLY CHAIN RISK ANALYSIS ===")
print(f"Top 4 Vendor Concentration (CR4): {risk_analysis['concentration_metrics']['CR4']:.2f}%")
print(f"Top 10 Vendor Concentration (CR10): {risk_analysis['concentration_metrics']['CR10']:.2f}%")
print(f"Herfindahl-Hirschman Index (HHI): {risk_analysis['concentration_metrics']['HHI']:.2f}")
print(f"Market Concentration Level: {risk_analysis['concentration_metrics']['Level']}")

print("\nRisk Distribution:")
print(risk_analysis['vendor_risk_scores']['RiskCategory'].value_counts())

In [ ]:
# Visualize Risk Analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Supply Chain Risk Analysis', fontsize=16, fontweight='bold')

# Vendor concentration
vendor_sales = df_enhanced.groupby('VendorName')['TotalSalesDollars'].sum().sort_values(ascending=False)
cumulative_concentration = (vendor_sales.cumsum() / vendor_sales.sum() * 100)

axes[0,0].plot(range(len(cumulative_concentration)), cumulative_concentration, linewidth=2)
axes[0,0].axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% Threshold')
axes[0,0].set_title('Vendor Concentration Curve')
axes[0,0].set_xlabel('Vendor Rank')
axes[0,0].set_ylabel('Cumulative Market Share (%)')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Risk category distribution
risk_counts = risk_analysis['vendor_risk_scores']['RiskCategory'].value_counts()
colors_risk = ['green', 'yellow', 'orange', 'red']
axes[0,1].pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%', 
               colors=colors_risk, startangle=90)
axes[0,1].set_title('Vendor Risk Distribution')

# Risk score vs sales
risk_data = risk_analysis['vendor_risk_scores']
scatter = axes[1,0].scatter(risk_data['TotalSalesDollars'], risk_data['TotalRiskScore'], 
                           c=risk_data['TotalRiskScore'], cmap='RdYlGn_r', alpha=0.7, s=50)
axes[1,0].set_xlabel('Total Sales Dollars ($)')
axes[1,0].set_ylabel('Risk Score')
axes[1,0].set_title('Risk Score vs Sales Volume')
axes[1,0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
axes[1,0].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[1,0])

# Top risk vendors
top_risk_vendors = risk_data.nlargest(10, 'TotalRiskScore')
axes[1,1].barh(range(len(top_risk_vendors)), top_risk_vendors['TotalRiskScore'], color='red')
axes[1,1].set_yticks(range(len(top_risk_vendors)))
axes[1,1].set_yticklabels(top_risk_vendors['VendorName'], fontsize=8)
axes[1,1].set_title('Top 10 High-Risk Vendors')
axes[1,1].set_xlabel('Risk Score')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Predictive Analytics and Forecasting

In [ ]:
# Vendor Performance Prediction
def vendor_performance_prediction(df):
    """Predict vendor performance and identify improvement opportunities"""
    
    # Create vendor-level features
    vendor_features = df.groupby('VendorName').agg({
        'TotalSalesDollars': 'sum',
        'TotalPurchaseDollars': 'sum',
        'GrossProfit': 'sum',
        'ProfitMargin': 'mean',
        'StockTurnover': 'mean',
        'FreightCost': 'sum',
        'TotalPurchaseQuantity': 'sum',
        'TotalSalesQuantity': 'sum',
        'Brand': 'nunique'
    }).reset_index()
    
    # Create performance indicators
    vendor_features['Profitability'] = vendor_features['GrossProfit'] / vendor_features['TotalSalesDollars']
    vendor_features['Efficiency'] = vendor_features['TotalSalesQuantity'] / vendor_features['TotalPurchaseQuantity']
    vendor_features['BrandDiversity'] = vendor_features['Brand']
    vendor_features['FreightRatio'] = vendor_features['FreightCost'] / vendor_features['TotalPurchaseDollars']
    
    # Performance scoring (0-100)
    vendor_features['PerformanceScore'] = (
        (vendor_features['Profitability'] * 40) +  # 40% weight
        (np.clip(vendor_features['Efficiency'], 0, 2) * 30) +  # 30% weight (capped at 2)
        (np.clip(vendor_features['BrandDiversity'] / 10, 0, 1) * 15) +  # 15% weight
        (np.clip(1 - vendor_features['FreightRatio'], 0, 1) * 15)  # 15% weight
    )
    
    # Performance categories
    vendor_features['PerformanceTier'] = pd.cut(vendor_features['PerformanceScore'], 
                                              bins=[0, 40, 60, 80, 100], 
                                              labels=['Poor', 'Average', 'Good', 'Excellent'])
    
    # Identify improvement opportunities
    improvement_opportunities = vendor_features[
        (vendor_features['PerformanceScore'] < 60) & 
        (vendor_features['TotalSalesDollars'] > vendor_features['TotalSalesDollars'].median())
    ].copy()
    
    return vendor_features, improvement_opportunities

vendor_performance, improvement_ops = vendor_performance_prediction(df_enhanced)

print("=== VENDOR PERFORMANCE PREDICTION ===")
print(f"Total vendors analyzed: {len(vendor_performance)}")
print(f"Vendors needing improvement: {len(improvement_ops)}")
print(f"Average performance score: {vendor_performance['PerformanceScore'].mean():.2f}")

print("\nPerformance Tier Distribution:")
print(vendor_performance['PerformanceTier'].value_counts())

In [ ]:
# Visualize Performance Prediction
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Vendor Performance Prediction & Improvement Opportunities', fontsize=16, fontweight='bold')

# Performance score distribution
axes[0,0].hist(vendor_performance['PerformanceScore'], bins=20, alpha=0.7, edgecolor='black', color='skyblue')
axes[0,0].axvline(vendor_performance['PerformanceScore'].mean(), color='red', linestyle='--', 
                 label=f'Mean: {vendor_performance["PerformanceScore"].mean():.1f}')
axes[0,0].set_title('Performance Score Distribution')
axes[0,0].set_xlabel('Performance Score')
axes[0,0].set_ylabel('Number of Vendors')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Performance tier pie chart
tier_counts = vendor_performance['PerformanceTier'].value_counts()
colors_tier = ['red', 'orange', 'lightgreen', 'darkgreen']
axes[0,1].pie(tier_counts.values, labels=tier_counts.index, autopct='%1.1f%%', 
               colors=colors_tier, startangle=90)
axes[0,1].set_title('Performance Tier Distribution')

# Performance components
performance_components = vendor_performance[['Profitability', 'Efficiency', 'BrandDiversity', 'FreightRatio']].mean()
components_normalized = (performance_components - performance_components.min()) / (performance_components.max() - performance_components.min())

angles = np.linspace(0, 2 * np.pi, len(components_normalized), endpoint=False).tolist()
angles += angles[:1]
values = components_normalized.tolist()
values += values[:1]

ax_radar = plt.subplot(2, 2, 3, projection='polar')
ax_radar.plot(angles, values, 'o-', linewidth=2, markersize=8)
ax_radar.fill(angles, values, alpha=0.25)
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(['Profitability', 'Efficiency', 'Brand Diversity', 'Cost Efficiency'])
ax_radar.set_title('Average Performance Components')

# Improvement opportunities
if len(improvement_ops) > 0:
    top_improvements = improvement_ops.nlargest(10, 'TotalSalesDollars')
    axes[1,1].barh(range(len(top_improvements)), top_improvements['PerformanceScore'], color='orange')
    axes[1,1].set_yticks(range(len(top_improvements)))
    axes[1,1].set_yticklabels(top_improvements['VendorName'], fontsize=8)
    axes[1,1].set_title('Top Improvement Opportunities')
    axes[1,1].set_xlabel('Performance Score')
    axes[1,1].grid(True, alpha=0.3)
else:
    axes[1,1].text(0.5, 0.5, 'No improvement opportunities\nidentified', 
                  ha='center', va='center', transform=axes[1,1].transAxes, fontsize=12)
    axes[1,1].set_title('Improvement Opportunities')

plt.tight_layout()
plt.show()

## 5. Strategic Business Insights Dashboard

In [ ]:
# Comprehensive Business Insights Summary
def generate_strategic_insights(df, vendor_perf, risk_analysis):
    """Generate comprehensive strategic business insights"""
    
    print("=" * 80)
    print("STRATEGIC BUSINESS INSIGHTS DASHBOARD")
    print("=" * 80)
    
    # 1. Market Overview
    total_sales = df['TotalSalesDollars'].sum()
    total_profit = df['GrossProfit'].sum()
    total_vendors = df['VendorName'].nunique()
    total_brands = df['Brand'].nunique()
    
    print(f"\n📊 MARKET OVERVIEW:")
    print(f"   • Total Revenue: ${total_sales:,.2f}")
    print(f"   • Total Profit: ${total_profit:,.2f}")
    print(f"   • Profit Margin: {(total_profit/total_sales)*100:.2f}%")
    print(f"   • Market Players: {total_vendors} vendors, {total_brands} brands")
    
    # 2. Vendor Landscape
    top_10_sales_share = (df.groupby('VendorName')['TotalSalesDollars'].sum().sort_values(ascending=False).head(10).sum() / total_sales) * 100
    avg_vendor_performance = vendor_perf['PerformanceScore'].mean()
    
    print(f"\n🏢 VENDOR LANDSCAPE:")
    print(f"   • Market Concentration: Top 10 vendors control {top_10_sales_share:.1f}% of sales")
    print(f"   • Average Vendor Performance: {avg_vendor_performance:.1f}/100")
    print(f"   • High-Risk Vendors: {len(risk_analysis['vendor_risk_scores'][risk_analysis['vendor_risk_scores']['RiskCategory'].isin(['High', 'Critical'])])}")
    
    # 3. Operational Efficiency
    avg_turnover = df['StockTurnover'].mean()
    avg_freight_ratio = (df['FreightCost'].sum() / df['TotalPurchaseDollars'].sum()) * 100
    unsold_inventory = ((df['TotalPurchaseQuantity'] - df['TotalSalesQuantity']) * (df['TotalPurchaseDollars'] / df['TotalPurchaseQuantity'])).sum()
    
    print(f"\n⚙️  OPERATIONAL EFFICIENCY:")
    print(f"   • Average Inventory Turnover: {avg_turnover:.2f}x")
    print(f"   • Freight Cost Ratio: {avg_freight_ratio:.2f}% of purchases")
    print(f"   • Capital in Unsold Inventory: ${unsold_inventory:,.2f}")
    
    # 4. Strategic Opportunities
    print(f"\n🎯 STRATEGIC OPPORTUNITIES:")
    
    # High-margin, low-sales brands
    brand_perf = df.groupby('Description').agg({
        'TotalSalesDollars': 'sum',
        'ProfitMargin': 'mean'
    })
    
    low_sales_threshold = brand_perf['TotalSalesDollars'].quantile(0.25)
    high_margin_threshold = brand_perf['ProfitMargin'].quantile(0.75)
    
    opportunity_brands = brand_perf[
        (brand_perf['TotalSalesDollars'] < low_sales_threshold) & 
        (brand_perf['ProfitMargin'] > high_margin_threshold)
    ]
    
    print(f"   • Brands for Promotion: {len(opportunity_brands)} brands with high margins but low sales")
    print(f"   • Vendor Improvement Needed: {len(improvement_ops)} vendors with potential for optimization")
    print(f"   • Bulk Purchasing Opportunity: Up to 72% cost reduction available for large orders")
    
    # 5. Risk Assessment
    hhi = risk_analysis['concentration_metrics']['HHI']
    critical_vendors = risk_analysis['vendor_risk_scores'][risk_analysis['vendor_risk_scores']['RiskCategory'] == 'Critical']
    
    print(f"\n⚠️  RISK ASSESSMENT:")
    print(f"   • Market Concentration Risk: {risk_analysis['concentration_metrics']['Level']} (HHI: {hhi:.1f})")
    print(f"   • Critical Risk Vendors: {len(critical_vendors)} vendors require immediate attention")
    print(f"   • Supply Chain Diversity: {'Adequate' if hhi < 1800 else 'Needs Improvement'}")
    
    return {
        'market_overview': {
            'total_sales': total_sales,
            'total_profit': total_profit,
            'profit_margin': (total_profit/total_sales)*100,
            'total_vendors': total_vendors
        },
        'vendor_landscape': {
            'concentration': top_10_sales_share,
            'avg_performance': avg_vendor_performance,
            'high_risk_count': len(risk_analysis['vendor_risk_scores'][risk_analysis['vendor_risk_scores']['RiskCategory'].isin(['High', 'Critical'])])
        },
        'operational_efficiency': {
            'avg_turnover': avg_turnover,
            'freight_ratio': avg_freight_ratio,
            'unsold_inventory': unsold_inventory
        }
    }

strategic_insights = generate_strategic_insights(df_enhanced, vendor_performance, risk_analysis)

In [ ]:
# Actionable Recommendations
def generate_actionable_recommendations(insights):
    """Generate specific actionable recommendations"""
    
    print("\n" + "=" * 80)
    print("ACTIONABLE RECOMMENDATIONS")
    print("=" * 80)
    
    print(f"\n🚀 IMMEDIATE ACTIONS (0-3 months):")
    print(f"   1. Address ${insights['operational_efficiency']['unsold_inventory']:,.2f} in unsold inventory")
    print(f"   2. Implement clearance sales for low-turnover products")
    print(f"   3. Review contracts with {insights['vendor_landscape']['high_risk_count']} high-risk vendors")
    print(f"   4. Optimize freight costs (currently {insights['operational_efficiency']['freight_ratio']:.2f}% of purchases)")
    
    print(f"\n📈 STRATEGIC INITIATIVES (3-12 months):")
    print(f"   1. Diversify vendor base to reduce concentration risk")
    print(f"   2. Implement vendor performance management system")
    print(f"   3. Develop promotional strategies for high-margin, low-sales brands")
    print(f"   4. Negotiate bulk purchasing discounts with top vendors")
    
    print(f"\n🎯 LONG-TERM TRANSFORMATION (12+ months):")
    print(f"   1. Implement predictive analytics for demand forecasting")
    print(f"   2. Develop vendor partnership programs")
    print(f"   3. Build supply chain resilience through redundancy")
    print(f"   4. Create data-driven pricing optimization engine")
    
    # Calculate potential impact
    potential_profit_increase = insights['market_overview']['total_profit'] * 0.15  # 15% improvement target
    cost_reduction_potential = insights['operational_efficiency']['unsold_inventory'] * 0.5  # 50% of unsold inventory
    
    print(f"\n💰 EXPECTED FINANCIAL IMPACT:")
    print(f"   • Potential Profit Increase: ${potential_profit_increase:,.2f}")
    print(f"   • Cost Reduction Opportunity: ${cost_reduction_potential:,.2f}")
    print(f"   • Total Financial Impact: ${potential_profit_increase + cost_reduction_potential:,.2f}")
    
    return {
        'profit_increase': potential_profit_increase,
        'cost_reduction': cost_reduction_potential,
        'total_impact': potential_profit_increase + cost_reduction_potential
    }

financial_impact = generate_actionable_recommendations(strategic_insights)

In [ ]:
# Save Enhanced Analysis Results
def save_enhanced_results(vendor_perf, risk_analysis, strategic_insights):
    """Save all enhanced analysis results for reporting"""
    
    # Save vendor performance predictions
    vendor_perf.to_csv('enhanced_vendor_performance.csv', index=False)
    
    # Save risk analysis
    risk_analysis['vendor_risk_scores'].to_csv('vendor_risk_assessment.csv', index=False)
    
    # Create executive summary
    executive_summary = pd.DataFrame({
        'Metric': ['Total Revenue', 'Total Profit', 'Profit Margin', 'Active Vendors', 
                  'Market Concentration', 'Avg Performance Score', 'Unsold Inventory', 
                  'Freight Cost Ratio', 'High-Risk Vendors', 'Potential Financial Impact'],
        'Value': [f"${strategic_insights['market_overview']['total_sales']:,.2f}",
                 f"${strategic_insights['market_overview']['total_profit']:,.2f}",
                 f"{strategic_insights['market_overview']['profit_margin']:.2f}%",
                 strategic_insights['market_overview']['total_vendors'],
                 f"{strategic_insights['vendor_landscape']['concentration']:.1f}%",
                 f"{strategic_insights['vendor_landscape']['avg_performance']:.1f}/100",
                 f"${strategic_insights['operational_efficiency']['unsold_inventory']:,.2f}",
                 f"{strategic_insights['operational_efficiency']['freight_ratio']:.2f}%",
                 strategic_insights['vendor_landscape']['high_risk_count'],
                 f"${financial_impact['total_impact']:,.2f}"]
    })
    
    executive_summary.to_csv('executive_summary.csv', index=False)
    
    print("✅ Enhanced analysis results saved:")
    print("   • enhanced_vendor_performance.csv")
    print("   • vendor_risk_assessment.csv")
    print("   • executive_summary.csv")
    
    return executive_summary

exec_summary_df = save_enhanced_results(vendor_performance, risk_analysis, strategic_insights)

print("\n" + "=" * 80)
print("🎉 ENHANCED BUSINESS ANALYSIS COMPLETE!")
print("=" * 80)
print(f"📊 Analysis Summary:")
print(f"   • {len(df_enhanced)} vendor-brand relationships analyzed")
print(f"   • {df_enhanced['VendorName'].nunique()} vendors segmented and scored")
print(f"   • Risk assessment completed for all vendors")
print(f"   • Performance predictions generated")
print(f"   • Strategic recommendations formulated")
print(f"\n💡 Key Insights:")
print(f"   • Market concentration: {strategic_insights['vendor_landscape']['concentration']:.1f}%")
print(f"   • Profit improvement potential: ${financial_impact['profit_increase']:,.2f}")
print(f"   • Cost reduction opportunity: ${financial_impact['cost_reduction']:,.2f}")
print(f"   • Total financial impact: ${financial_impact['total_impact']:,.2f}")